In [1]:
import os, glob
from collections import defaultdict, Counter
import polars as pl

hits = glob.glob('/kaggle/input/**/train_base.parquet', recursive=True)
TRAIN = os.path.dirname(hits[0])

groups = defaultdict(list)
for f in sorted(os.listdir(TRAIN)):
    tok = f.removeprefix('train_').removesuffix('.parquet').split('_')
    key = '_'.join(tok[:-1]) if tok[-1].isdigit() and tok[-2].isdigit() else '_'.join(tok)
    groups[key].append(f'{TRAIN}/{f}')

DEPTH1 = ['applprev_1', 'credit_bureau_a_1', 'credit_bureau_b_1', 'person_1',
          'tax_registry_a_1', 'tax_registry_b_1', 'tax_registry_c_1',
          'debitcard_1', 'deposit_1', 'other_1']

for k in DEPTH1:
    lf = pl.scan_parquet(groups[k])
    names = lf.collect_schema().names()
    feats = [c for c in names if c not in ('case_id', 'num_group1')]
    rows, cases, max_ng1 = lf.select(
        pl.len(), pl.col('case_id').n_unique(), pl.col('num_group1').max()
    ).collect().row(0)
    print(f'{k:20s} rows={rows:>11,} cases={cases:>10,} max_num_group1={max_ng1:>4} '
          f'suffix={dict(Counter(c[-1] for c in feats))}')

applprev_1           rows=  6,525,979 cases= 1,221,522 max_num_group1=  19 suffix={'P': 2, 'A': 11, 'D': 7, 'L': 12, 'M': 7}
credit_bureau_a_1    rows= 15,940,537 cases= 1,386,273 max_num_group1= 528 suffix={'L': 20, 'M': 11, 'A': 24, 'D': 12, 'P': 2, 'T': 8}
credit_bureau_b_1    rows=     85,791 cases=    36,500 max_num_group1=  20 suffix={'A': 16, 'M': 9, 'D': 3, 'L': 7, 'P': 4, 'T': 4}
person_1             rows=  2,973,991 cases= 1,526,659 max_num_group1=   9 suffix={'D': 3, 'L': 20, 'M': 8, 'T': 3, 'A': 1}
tax_registry_a_1     rows=  3,275,770 cases=   457,934 max_num_group1=  98 suffix={'A': 1, 'M': 1, 'D': 1}
tax_registry_b_1     rows=  1,107,933 cases=   150,732 max_num_group1= 100 suffix={'A': 1, 'D': 1, 'M': 1}
tax_registry_c_1     rows=  3,343,800 cases=   482,265 max_num_group1= 120 suffix={'M': 1, 'A': 1, 'D': 1}
debitcard_1          rows=    157,302 cases=   111,772 max_num_group1=  65 suffix={'A': 3, 'D': 1}
deposit_1            rows=    145,086 cases=   105,111 max_num_g

In [2]:
# 1) Các phần của cùng một bảng có cùng schema không
merged = {}
for k in DEPTH1:
    schemas = [pl.read_parquet_schema(p) for p in groups[k]]
    sets = [set(s) for s in schemas]
    union, inter = set().union(*sets), set.intersection(*sets)
    merged[k] = {c: t for s in schemas for c, t in s.items()}
    if len(groups[k]) > 1:
        print(f'{k:20s} parts={len(groups[k])} union={len(union)} intersection={len(inter)}')

# 2) Kiểu dữ liệu theo hậu tố
print()
for k in DEPTH1:
    dt = Counter((c[-1], str(t)) for c, t in merged[k].items() if c not in ('case_id', 'num_group1'))
    print(f'{k:20s}', dict(dt))

# 3) person_1: mỗi hồ sơ có đúng một dòng num_group1 = 0 không
print()
p0 = pl.scan_parquet(groups['person_1']).filter(pl.col('num_group1') == 0)
print(p0.select(pl.len(), pl.col('case_id').n_unique()).collect())

applprev_1           parts=2 union=41 intersection=41
credit_bureau_a_1    parts=4 union=79 intersection=79

applprev_1           {('P', 'Float64'): 2, ('A', 'Float64'): 11, ('D', 'String'): 7, ('L', 'Float64'): 5, ('M', 'String'): 7, ('L', 'String'): 5, ('L', 'Boolean'): 2}
credit_bureau_a_1    {('L', 'Float64'): 20, ('M', 'String'): 11, ('A', 'Float64'): 24, ('D', 'String'): 12, ('P', 'Float64'): 2, ('T', 'Float64'): 8}
credit_bureau_b_1    {('A', 'Float64'): 16, ('M', 'String'): 9, ('D', 'String'): 3, ('L', 'Float64'): 6, ('P', 'Float64'): 4, ('T', 'Float64'): 4, ('L', 'String'): 1}
person_1             {('D', 'String'): 3, ('L', 'Float64'): 4, ('M', 'String'): 8, ('L', 'Boolean'): 5, ('L', 'String'): 11, ('T', 'String'): 3, ('A', 'Float64'): 1}
tax_registry_a_1     {('A', 'Float64'): 1, ('M', 'String'): 1, ('D', 'String'): 1}
tax_registry_b_1     {('A', 'Float64'): 1, ('D', 'String'): 1, ('M', 'String'): 1}
tax_registry_c_1     {('M', 'String'): 1, ('A', 'Float64'): 1, ('D', 'Strin

In [3]:
def to_date(c, dtype):
    return pl.col(c).str.to_date(strict=False) if dtype == pl.String else pl.col(c).cast(pl.Date, strict=False)

b = pl.scan_parquet(groups['base'])
base_dates = b.select('case_id', to_date('date_decision', b.collect_schema()['date_decision']))

def agg_table(name, lf):
    sch = lf.collect_schema()
    feats = [c for c in sch.names() if c not in ('case_id', 'num_group1', 'num_group2')]
    dcols = [c for c in feats if c[-1] == 'D']
    lf = lf.join(base_dates, on='case_id', how='left').with_columns(
        [(to_date(c, sch[c]) - pl.col('date_decision')).dt.total_days().cast(pl.Float32).alias(c) for c in dcols])
    aggs = [pl.len().alias(f'{name}__count')]
    for c in feats:
        t, p = sch[c], f'{name}__{c}'
        if c in dcols:
            aggs += [pl.col(c).min().alias(f'{p}__min'), pl.col(c).max().alias(f'{p}__max')]
        elif t == pl.Boolean:
            aggs.append(pl.col(c).cast(pl.Float32).mean().alias(f'{p}__mean'))
        elif t.is_numeric():
            x = pl.col(c).cast(pl.Float32)
            aggs += [x.max().alias(f'{p}__max'), x.mean().alias(f'{p}__mean')]
            if c[-1] == 'A':
                aggs.append(pl.when(x.count() > 0).then(x.sum()).alias(f'{p}__sum'))
        else:
            aggs.append(pl.col(c).drop_nulls().n_unique().alias(f'{p}__nuniq'))
    return lf.group_by('case_id').agg(aggs)

for k in ['debitcard_1', 'tax_registry_a_1']:
    out = agg_table(k, pl.scan_parquet(groups[k])).collect()
    print(k, out.shape)
    print(out.head(3))

debitcard_1 (111772, 13)
shape: (3, 13)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ case_id ┆ debitcard_ ┆ debitcard ┆ debitcard ┆ … ┆ debitcard ┆ debitcard ┆ debitcard ┆ debitcard │
│ ---     ┆ 1__count   ┆ _1__last1 ┆ _1__last1 ┆   ┆ _1__last3 ┆ _1__last3 ┆ _1__openi ┆ _1__openi │
│ i64     ┆ ---        ┆ 80dayaver ┆ 80dayaver ┆   ┆ 0dayturno ┆ 0dayturno ┆ ngdate_85 ┆ ngdate_85 │
│         ┆ u32        ┆ age…      ┆ age…      ┆   ┆ ver…      ┆ ver…      ┆ 7D_…      ┆ 7D_…      │
│         ┆            ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│         ┆            ┆ f32       ┆ f32       ┆   ┆ f32       ┆ f32       ┆ f32       ┆ f32       │
╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1821451 ┆ 1          ┆ null      ┆ null      ┆ … ┆ null      ┆ null      ┆ -2340.0   ┆ -2340.0   │
│ 636354  ┆ 1          ┆ null      ┆ null      ┆ … 

In [4]:
rows = []
for k in DEPTH1:
    lf = pl.scan_parquet(groups[k])
    sch = lf.collect_schema()
    dcols = [c for c in sch.names() if c[-1] == 'D']
    if not dcols:
        continue
    lf = lf.join(base_dates, on='case_id', how='left')
    exprs = []
    for c in dcols:
        d = (to_date(c, sch[c]) - pl.col('date_decision')).dt.total_days()
        exprs += [(d > 0).mean().alias(f'{c}|pos'), d.min().alias(f'{c}|min'),
                  d.median().alias(f'{c}|med'), d.max().alias(f'{c}|max'),
                  d.is_null().mean().alias(f'{c}|null')]
    r = lf.select(exprs).collect().row(0, named=True)
    for c in dcols:
        rows.append({'table': k, 'col': c, 'pos_rate': r[f'{c}|pos'], 'min': r[f'{c}|min'],
                     'median': r[f'{c}|med'], 'max': r[f'{c}|max'], 'null_rate': r[f'{c}|null']})

res = pl.DataFrame(rows).sort('pos_rate', descending=True, nulls_last=True)
with pl.Config(tbl_rows=60, tbl_width_chars=200):
    print(res)

shape: (31, 7)
┌───────────────────┬─────────────────────────────────┬──────────┬────────┬──────────┬───────┬───────────┐
│ table             ┆ col                             ┆ pos_rate ┆ min    ┆ median   ┆ max   ┆ null_rate │
│ ---               ┆ ---                             ┆ ---      ┆ ---    ┆ ---      ┆ ---   ┆ ---       │
│ str               ┆ str                             ┆ f64      ┆ i64    ┆ f64      ┆ i64   ┆ f64       │
╞═══════════════════╪═════════════════════════════════╪══════════╪════════╪══════════╪═══════╪═══════════╡
│ tax_registry_a_1  ┆ recorddate_4527225D             ┆ 1.0      ┆ 3      ┆ 14.0     ┆ 14    ┆ 0.0       │
│ credit_bureau_a_1 ┆ dateofcredend_289D              ┆ 0.989146 ┆ -5737  ┆ 436.0    ┆ 28697 ┆ 0.833175  │
│ credit_bureau_b_1 ┆ contractmaturitydate_151D       ┆ 0.971204 ┆ -5911  ┆ 424.0    ┆ 9138  ┆ 0.047546  │
│ credit_bureau_b_1 ┆ lastupdate_260D                 ┆ 0.888326 ┆ -2013  ┆ 8.0      ┆ 14    ┆ 0.045366  │
│ credit_bureau_a_1 ┆ 

In [5]:
DROP_D = set(res.filter((pl.col('median') > 0) & (pl.col('max') <= 14))['col'])
KEEP_FUTURE_D = {'dateofcredend_289D', 'contractmaturitydate_151D', 'dateofcredend_353D',
                 'contractenddate_991D', 'firstnonzeroinstldate_307D'}
ROW_FILTER = {'applprev_1': 'creationdate_885D',
              'tax_registry_b_1': 'deductiondate_4917603D',
              'tax_registry_c_1': 'processingdate_168D'}
print('DROP_D:', DROP_D)

def agg_table(name, lf):
    sch = lf.collect_schema()
    feats = [c for c in sch.names() if c not in ('case_id', 'num_group1', 'num_group2') and c not in DROP_D]
    dcols = [c for c in feats if c[-1] == 'D']
    lf = lf.join(base_dates, on='case_id', how='left').with_columns(
        [(to_date(c, sch[c]) - pl.col('date_decision')).dt.total_days().cast(pl.Float32).alias(c) for c in dcols])
    if name in ROW_FILTER:                                   # nhóm 3: bỏ dòng sự kiện sau quyết định
        fc = ROW_FILTER[name]
        lf = lf.filter(pl.col(fc).is_null() | (pl.col(fc) <= 0))
    lf = lf.with_columns([pl.when(pl.col(c) > 0).then(pl.lit(None)).otherwise(pl.col(c)).alias(c)
                          for c in dcols if c not in KEEP_FUTURE_D])   # nhóm 4: giá trị > 0 → null
    aggs = [pl.len().alias(f'{name}__count')]
    for c in feats:
        t, p = sch[c], f'{name}__{c}'
        if c in dcols:
            aggs += [pl.col(c).min().alias(f'{p}__min'), pl.col(c).max().alias(f'{p}__max')]
        elif t == pl.Boolean:
            aggs.append(pl.col(c).cast(pl.Float32).mean().alias(f'{p}__mean'))
        elif t.is_numeric():
            x = pl.col(c).cast(pl.Float32)
            aggs += [x.max().alias(f'{p}__max'), x.mean().alias(f'{p}__mean')]
            if c[-1] == 'A':
                aggs.append(pl.when(x.count() > 0).then(x.sum()).alias(f'{p}__sum'))
        else:
            aggs.append(pl.col(c).drop_nulls().n_unique().alias(f'{p}__nuniq'))
    return lf.group_by('case_id').agg(aggs)

# Kiểm tra: sau xử lý, cột ngày nào còn giá trị > 0?
for k in ['tax_registry_a_1', 'tax_registry_c_1', 'applprev_1']:
    out = agg_table(k, pl.scan_parquet(groups[k])).collect()
    pos = {c: out[c].max() for c in out.columns
           if c.endswith('__max') and c.split('__')[1].endswith('D') and (out[c].max() or 0) > 0}
    print(k, out.shape, '| cột ngày còn > 0:', pos)


DROP_D: {'lastupdate_1112D', 'refreshdate_3813885D', 'lastupdate_260D', 'recorddate_4527225D'}
tax_registry_a_1 (457934, 6) | cột ngày còn > 0: {}
tax_registry_c_1 (403018, 8) | cột ngày còn > 0: {}
applprev_1 (1208039, 77) | cột ngày còn > 0: {'applprev_1__firstnonzeroinstldate_307D__max': 45.0}


In [6]:
def person_features():
    lf = pl.scan_parquet(groups['person_1'])
    sch = lf.collect_schema()
    feats = [c for c in sch.names() if c not in ('case_id', 'num_group1') and c not in DROP_D]
    dcols = [c for c in feats if c[-1] == 'D']
    app = (lf.filter(pl.col('num_group1') == 0)
             .join(base_dates, on='case_id', how='left')
             .with_columns([(to_date(c, sch[c]) - pl.col('date_decision')).dt.total_days().cast(pl.Float32).alias(c) for c in dcols])
             .with_columns([pl.when(pl.col(c) > 0).then(pl.lit(None)).otherwise(pl.col(c)).alias(c) for c in dcols])
             .with_columns([pl.col(c).cast(pl.Float32) for c in feats if c not in dcols and sch[c].is_numeric()])
             .select(['case_id'] + [pl.col(c).alias(f'person_1__app__{c}') for c in feats]))
    rel = (lf.filter(pl.col('num_group1') > 0)
             .group_by('case_id').agg(pl.len().alias('person_1__n_related')))
    return (app.join(rel, on='case_id', how='left')
               .with_columns(pl.col('person_1__n_related').fill_null(0)))

def other_features():
    lf = pl.scan_parquet(groups['other_1'])
    feats = [c for c in lf.collect_schema().names() if c not in ('case_id', 'num_group1')]
    return lf.select(['case_id'] + [pl.col(c).cast(pl.Float32).alias(f'other_1__{c}') for c in feats])

per = person_features().collect()
oth = other_features().collect()
print('person_1', per.shape, '| other_1', oth.shape)
print(per['person_1__n_related'].value_counts().sort('person_1__n_related'))
print(dict(Counter(str(t) for t in per.schema.values())))

person_1 (1526659, 37) | other_1 (51109, 6)
shape: (9, 2)
┌─────────────────────┬────────┐
│ person_1__n_related ┆ count  │
│ ---                 ┆ ---    │
│ u32                 ┆ u32    │
╞═════════════════════╪════════╡
│ 0                   ┆ 769339 │
│ 1                   ┆ 273106 │
│ 2                   ┆ 302446 │
│ 3                   ┆ 159315 │
│ 4                   ┆ 20987  │
│ 5                   ┆ 1367   │
│ 6                   ┆ 91     │
│ 7                   ┆ 6      │
│ 9                   ┆ 2      │
└─────────────────────┴────────┘
{'Int64': 1, 'Float32': 8, 'String': 22, 'Boolean': 5, 'UInt32': 1}


In [7]:
import time

parts = {k: agg_table(k, pl.scan_parquet(groups[k]))
         for k in DEPTH1 if k not in ('person_1', 'other_1')}
parts['person_1'] = person_features()
parts['other_1'] = other_features()

df1 = pl.scan_parquet(groups['base']).select('case_id')
for k, lf in parts.items():
    df1 = df1.join(lf, on='case_id', how='left')

t = time.time()
out1 = df1.collect()
print(out1.shape, f'{out1.estimated_size("gb"):.2f} GB', f'{time.time() - t:.0f}s')
print('case_id unique:', out1['case_id'].n_unique() == out1.height)
print(dict(Counter(c.split('__')[0] for c in out1.columns[1:])))

out1.write_parquet('/kaggle/working/depth1_agg.parquet')

(1526659, 412) 2.51 GB 66s
case_id unique: True
{'applprev_1': 76, 'credit_bureau_a_1': 164, 'credit_bureau_b_1': 91, 'tax_registry_a_1': 5, 'tax_registry_b_1': 7, 'tax_registry_c_1': 7, 'debitcard_1': 12, 'deposit_1': 8, 'person_1': 36, 'other_1': 5}


In [8]:
# === Chẩn đoán trước khi sửa depth 1 ===
# (1) Cặp cột năm–tháng và cột giá trị đi kèm: so mẫu null theo từng dòng
for k in ['credit_bureau_a_1', 'credit_bureau_b_1']:
    lf = pl.scan_parquet(groups[k])
    sch = lf.collect_schema()
    years = [c for c in sch.names() if c.endswith('T') and 'year' in c]
    months = [c for c in sch.names() if c.endswith('T') and 'month' in c]
    vals = [c for c in sch.names() if c[-1] in 'PA' and ('dpdmax' in c or 'overdueamountmax' in c)]
    s = lf.select(years + months + vals).collect()
    nn = {c: s[c].is_not_null() for c in s.columns}
    agree = lambda a, b: round(float((nn[a] == nn[b]).mean()), 4)
    print(f'\n=== {k} | {s.height:,} dòng ===')
    print('T còn lại (không phải năm/tháng):', [c for c in sch.names() if c.endswith('T') and c not in years + months])
    print('Tỷ lệ có giá trị:', {c: round(float(nn[c].mean()), 3) for c in s.columns})
    print('Năm ~ tháng:')
    for y in years:
        print('  ', y, {m: agree(y, m) for m in months})
    print('Năm ~ cột giá trị:')
    for y in years:
        print('  ', y, {v: agree(y, v) for v in vals})
    print('Khoảng giá trị năm/tháng:', {c: (s[c].min(), s[c].max()) for c in years + months})

# (2) Tỷ lệ mã thiếu 'a55475b1' trong các cột M của depth 1 (chỉ in cột > 0)
print('\n=== a55475b1 ===')
for k in DEPTH1:
    lf = pl.scan_parquet(groups[k])
    mcols = [c for c in lf.collect_schema().names() if c.endswith('M')]
    if not mcols:
        continue
    r = lf.select([(pl.col(c) == 'a55475b1').mean().alias(c) for c in mcols]).collect().row(0, named=True)
    print(k, {c: round(v, 3) for c, v in r.items() if v and v > 0})


=== credit_bureau_a_1 | 15,940,537 dòng ===
T còn lại (không phải năm/tháng): []
Tỷ lệ có giá trị: {'dpdmaxdateyear_596T': 0.166, 'dpdmaxdateyear_896T': 0.409, 'overdueamountmaxdateyear_2T': 0.167, 'overdueamountmaxdateyear_994T': 0.41, 'dpdmaxdatemonth_442T': 0.409, 'dpdmaxdatemonth_89T': 0.166, 'overdueamountmaxdatemonth_284T': 0.41, 'overdueamountmaxdatemonth_365T': 0.167, 'dpdmax_139P': 0.166, 'dpdmax_757P': 0.409, 'overdueamountmax2_14A': 0.167, 'overdueamountmax2_398A': 0.424, 'overdueamountmax_155A': 0.167, 'overdueamountmax_35A': 0.41}
Năm ~ tháng:
   dpdmaxdateyear_596T {'dpdmaxdatemonth_442T': 0.6458, 'dpdmaxdatemonth_89T': 1.0, 'overdueamountmaxdatemonth_284T': 0.6457, 'overdueamountmaxdatemonth_365T': 0.999}
   dpdmaxdateyear_896T {'dpdmaxdatemonth_442T': 1.0, 'dpdmaxdatemonth_89T': 0.6458, 'overdueamountmaxdatemonth_284T': 0.9995, 'overdueamountmaxdatemonth_365T': 0.6462}
   overdueamountmaxdateyear_2T {'dpdmaxdatemonth_442T': 0.6462, 'dpdmaxdatemonth_89T': 0.999, 'overdu

In [9]:
# === Chẩn đoán 2: tỷ lệ sự kiện max DPD / quá hạn ở tháng >= tháng quyết định ===
PAIRS = {
    'credit_bureau_a_1': [('dpdmaxdateyear_596T', 'dpdmaxdatemonth_89T', 'dpdmax_139P'),
                          ('dpdmaxdateyear_896T', 'dpdmaxdatemonth_442T', 'dpdmax_757P'),
                          ('overdueamountmaxdateyear_2T', 'overdueamountmaxdatemonth_365T', 'overdueamountmax_155A'),
                          ('overdueamountmaxdateyear_994T', 'overdueamountmaxdatemonth_284T', 'overdueamountmax_35A')],
    'credit_bureau_b_1': [('dpdmaxdateyear_742T', 'dpdmaxdatemonth_804T', 'dpdmax_851P'),
                          ('overdueamountmaxdateyear_432T', 'overdueamountmaxdatemonth_494T', 'overdueamountmax_950A')],
}
b = pl.scan_parquet(groups['base'])
dd = pl.col('date_decision')
dd = dd.str.to_date(strict=False) if b.collect_schema()['date_decision'] == pl.String else dd.cast(pl.Date)
base_ym = b.select('case_id', (dd.dt.year().cast(pl.Int32) * 12 + dd.dt.month().cast(pl.Int32) - 1).alias('dec_m'))

for k, pairs in PAIRS.items():
    lf = pl.scan_parquet(groups[k]).join(base_ym, on='case_id', how='left')
    ex = []
    for y, m, v in pairs:
        ago = pl.col('dec_m') - pl.when(pl.col(y) >= 1980).then(pl.col(y) * 12 + pl.col(m) - 1)
        ex += [ago.is_not_null().sum().alias(f'{y}|n'), (ago == 0).sum().alias(f'{y}|same'),
               (ago < 0).sum().alias(f'{y}|after'), (pl.col(y) < 1980).sum().alias(f'{y}|bad'),
               ago.min().alias(f'{y}|min'), ago.median().alias(f'{y}|med'),
               (pl.col(v).is_not_null() & (ago <= 0)).sum().alias(f'{y}|vmask')]
    r = lf.select(ex).collect().row(0, named=True)
    print(f'\n=== {k} ===')
    for y, m, v in pairs:
        n = r[f'{y}|n']
        print(f"{v:22s} | n {n:>10,} | cùng tháng QĐ {r[f'{y}|same'] / n:.4f} | sau tháng QĐ {r[f'{y}|after'] / n:.4f} "
              f"| năm < 1980: {r[f'{y}|bad']:,} | số tháng trước QĐ: min {r[f'{y}|min']}, median {r[f'{y}|med']} "
              f"| giá trị sẽ bị che: {r[f'{y}|vmask']:,}")


=== credit_bureau_a_1 ===
dpdmax_139P            | n  2,644,036 | cùng tháng QĐ 0.0679 | sau tháng QĐ 0.0310 | năm < 1980: 0 | số tháng trước QĐ: min -1.0, median 8.0 | giá trị sẽ bị che: 261,453
dpdmax_757P            | n  6,526,895 | cùng tháng QĐ 0.0044 | sau tháng QĐ 0.0011 | năm < 1980: 0 | số tháng trước QĐ: min -1.0, median 44.0 | giá trị sẽ bị che: 36,274
overdueamountmax_155A  | n  2,659,275 | cùng tháng QĐ 0.0654 | sau tháng QĐ 0.0257 | năm < 1980: 0 | số tháng trước QĐ: min -1.0, median 8.0 | giá trị sẽ bị che: 242,030
overdueamountmax_35A   | n  6,535,045 | cùng tháng QĐ 0.0038 | sau tháng QĐ 0.0010 | năm < 1980: 0 | số tháng trước QĐ: min -1.0, median 45.0 | giá trị sẽ bị che: 31,255

=== credit_bureau_b_1 ===
dpdmax_851P            | n     81,223 | cùng tháng QĐ 0.0809 | sau tháng QĐ 0.0349 | năm < 1980: 1 | số tháng trước QĐ: min -1.0, median 7.0 | giá trị sẽ bị che: 9,409
overdueamountmax_950A  | n     81,223 | cùng tháng QĐ 0.0916 | sau tháng QĐ 0.0521 | năm < 1980: 1

In [10]:
# === depth 1 v2: a55475b1 -> null; năm+tháng -> số tháng trước QĐ; che max DPD/quá hạn ở tháng >= tháng QĐ ===
import time
from collections import Counter

DROP_D = {'recorddate_4527225D', 'lastupdate_260D', 'refreshdate_3813885D', 'lastupdate_1112D'}
KEEP_FUTURE_D = {'dateofcredend_289D', 'contractmaturitydate_151D', 'dateofcredend_353D',
                 'contractenddate_991D', 'firstnonzeroinstldate_307D'}
ROW_FILTER = {'applprev_1': 'creationdate_885D', 'tax_registry_b_1': 'deductiondate_4917603D',
              'tax_registry_c_1': 'processingdate_168D'}
PAIRS = {
    'credit_bureau_a_1': [('dpdmaxdateyear_596T', 'dpdmaxdatemonth_89T', 'dpdmax_139P'),
                          ('dpdmaxdateyear_896T', 'dpdmaxdatemonth_442T', 'dpdmax_757P'),
                          ('overdueamountmaxdateyear_2T', 'overdueamountmaxdatemonth_365T', 'overdueamountmax_155A'),
                          ('overdueamountmaxdateyear_994T', 'overdueamountmaxdatemonth_284T', 'overdueamountmax_35A')],
    'credit_bureau_b_1': [('dpdmaxdateyear_742T', 'dpdmaxdatemonth_804T', 'dpdmax_851P'),
                          ('overdueamountmaxdateyear_432T', 'overdueamountmaxdatemonth_494T', 'overdueamountmax_950A')],
}
NA = 'a55475b1'

def to_date(c, dtype):
    return pl.col(c).str.to_date(strict=False) if dtype == pl.String else pl.col(c).cast(pl.Date, strict=False)

b = pl.scan_parquet(groups['base'])
dd = to_date('date_decision', b.collect_schema()['date_decision'])
base_dates = b.select('case_id', dd.alias('date_decision'),
                      (dd.dt.year().cast(pl.Int32) * 12 + dd.dt.month().cast(pl.Int32) - 1).alias('dec_m'))

def agg_table_v2(name, lf):
    sch = lf.collect_schema()
    pairs = PAIRS.get(name, [])
    ym = {c for y, m, v in pairs for c in (y, m)}
    feats = [c for c in sch.names() if c not in ('case_id', 'num_group1', 'num_group2')
             and c not in DROP_D and c not in ym]
    dcols = [c for c in feats if c[-1] == 'D']
    scols = [c for c in feats if sch[c] == pl.String and c[-1] != 'D']
    lf = (lf.join(base_dates, on='case_id', how='left')
            .with_columns([pl.when(pl.col(c) == NA).then(pl.lit(None)).otherwise(pl.col(c)).alias(c) for c in scols])
            .with_columns([(to_date(c, sch[c]) - pl.col('date_decision')).dt.total_days().cast(pl.Float32).alias(c)
                           for c in dcols]))
    if name in ROW_FILTER:
        fc = ROW_FILTER[name]
        lf = lf.filter(pl.col(fc).is_null() | (pl.col(fc) <= 0))
    lf = lf.with_columns([pl.when(pl.col(c) > 0).then(pl.lit(None)).otherwise(pl.col(c)).alias(c)
                          for c in dcols if c not in KEEP_FUTURE_D])
    mago = [f'{v}__mago' for y, m, v in pairs]
    if pairs:
        lf = lf.with_columns([(pl.col('dec_m') - pl.when(pl.col(y) >= 1980).then(pl.col(y) * 12 + pl.col(m) - 1))
                              .cast(pl.Float32).alias(f'{v}__mago') for y, m, v in pairs])
        lf = lf.with_columns([pl.when(pl.col(f'{v}__mago') <= 0).then(pl.lit(None)).otherwise(pl.col(c)).alias(c)
                              for y, m, v in pairs for c in (v, f'{v}__mago')])
    aggs = [pl.len().alias(f'{name}__count')]
    for c in feats:
        t, p = sch[c], f'{name}__{c}'
        if c in dcols:
            aggs += [pl.col(c).min().alias(f'{p}__min'), pl.col(c).max().alias(f'{p}__max')]
        elif t == pl.Boolean:
            aggs.append(pl.col(c).cast(pl.Float32).mean().alias(f'{p}__mean'))
        elif t.is_numeric():
            x = pl.col(c).cast(pl.Float32)
            aggs += [x.max().alias(f'{p}__max'), x.mean().alias(f'{p}__mean')]
            if c[-1] == 'A':
                aggs.append(pl.when(x.count() > 0).then(x.sum()).alias(f'{p}__sum'))
        else:
            aggs.append(pl.col(c).drop_nulls().n_unique().alias(f'{p}__nuniq'))
    for c in mago:
        aggs += [pl.col(c).min().alias(f'{name}__{c}__min'), pl.col(c).max().alias(f'{name}__{c}__max')]
    return lf.group_by('case_id').agg(aggs)

def person_features():   # giữ nguyên như bản cũ
    lf = pl.scan_parquet(groups['person_1'])
    sch = lf.collect_schema()
    feats = [c for c in sch.names() if c not in ('case_id', 'num_group1') and c not in DROP_D]
    dcols = [c for c in feats if c[-1] == 'D']
    app = (lf.filter(pl.col('num_group1') == 0)
             .join(base_dates.select('case_id', 'date_decision'), on='case_id', how='left')
             .with_columns([(to_date(c, sch[c]) - pl.col('date_decision')).dt.total_days().cast(pl.Float32).alias(c) for c in dcols])
             .with_columns([pl.when(pl.col(c) > 0).then(pl.lit(None)).otherwise(pl.col(c)).alias(c) for c in dcols])
             .with_columns([pl.col(c).cast(pl.Float32) for c in feats if c not in dcols and sch[c].is_numeric()])
             .select(['case_id'] + [pl.col(c).alias(f'person_1__app__{c}') for c in feats]))
    rel = lf.filter(pl.col('num_group1') > 0).group_by('case_id').agg(pl.len().alias('person_1__n_related'))
    return app.join(rel, on='case_id', how='left').with_columns(pl.col('person_1__n_related').fill_null(0))

def other_features():    # giữ nguyên như bản cũ
    lf = pl.scan_parquet(groups['other_1'])
    feats = [c for c in lf.collect_schema().names() if c not in ('case_id', 'num_group1')]
    return lf.select(['case_id'] + [pl.col(c).cast(pl.Float32).alias(f'other_1__{c}') for c in feats])

parts = {k: agg_table_v2(k, pl.scan_parquet(groups[k])) for k in DEPTH1 if k not in ('person_1', 'other_1')}
parts['person_1'] = person_features()
parts['other_1'] = other_features()
df1 = pl.scan_parquet(groups['base']).select('case_id')
for k, lf in parts.items():
    df1 = df1.join(lf, on='case_id', how='left')

t = time.time()
out2 = df1.collect()
print(out2.shape, f'{out2.estimated_size("gb"):.2f} GB', f'{time.time() - t:.0f}s')
assert out2['case_id'].n_unique() == out2.height
print(dict(Counter(c.split('__')[0] for c in out2.columns[1:])))
mcols = [c for c in out2.columns if '__mago__' in c]
print('Cột mới:', mcols)
print('Còn cột năm/tháng:', [c for c in out2.columns if 'dateyear' in c or 'datemonth' in c])
print('mago nhỏ nhất (phải >= 1):', out2.select([pl.col(c).min() for c in mcols]).row(0))
out2.write_parquet('/kaggle/working/depth1_agg_v2.parquet')

(1526659, 400) 2.44 GB 65s
{'applprev_1': 76, 'credit_bureau_a_1': 156, 'credit_bureau_b_1': 87, 'tax_registry_a_1': 5, 'tax_registry_b_1': 7, 'tax_registry_c_1': 7, 'debitcard_1': 12, 'deposit_1': 8, 'person_1': 36, 'other_1': 5}
Cột mới: ['credit_bureau_a_1__dpdmax_139P__mago__min', 'credit_bureau_a_1__dpdmax_139P__mago__max', 'credit_bureau_a_1__dpdmax_757P__mago__min', 'credit_bureau_a_1__dpdmax_757P__mago__max', 'credit_bureau_a_1__overdueamountmax_155A__mago__min', 'credit_bureau_a_1__overdueamountmax_155A__mago__max', 'credit_bureau_a_1__overdueamountmax_35A__mago__min', 'credit_bureau_a_1__overdueamountmax_35A__mago__max', 'credit_bureau_b_1__dpdmax_851P__mago__min', 'credit_bureau_b_1__dpdmax_851P__mago__max', 'credit_bureau_b_1__overdueamountmax_950A__mago__min', 'credit_bureau_b_1__overdueamountmax_950A__mago__max']
Còn cột năm/tháng: []
mago nhỏ nhất (phải >= 1): (1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0)
